In [12]:
import pandas as pd
import plotly.express as px

In [2]:
df = pd.read_csv("../data/raw/complaints.csv")
df.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
0,2020-07-06,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,FL,346XX,NaN,Other,Web,2020-07-06,Closed with explanation,Yes,NaN,3730948
1,2019-12-26,Credit card or prepaid card,General-purpose credit card or charge card,"Advertising and marketing, including promotion...",Confusing or misleading advertising about the ...,NaN,NaN,CAPITAL ONE FINANCIAL CORPORATION,CA,94025,NaN,Consent not provided,Web,2019-12-26,Closed with explanation,Yes,NaN,3477549
2,2020-05-08,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,These are not my accounts.,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,NV,89030,NaN,Consent provided,Web,2020-05-08,Closed with explanation,Yes,NaN,3642453
3,2024-01-05,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,Kindly address this issue on my credit report....,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,IL,60502,NaN,Consent provided,Web,2024-01-05,Closed with non-monetary relief,Yes,NaN,8113747
4,2024-01-21,Credit reporting or other personal consumer re...,Credit reporting,Improper use of your report,Credit inquiries on your report that you don't...,NaN,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,NC,27401,Servicemember,Consent not provided,Web,2024-01-21,Closed with explanation,Yes,NaN,8191825


In [3]:
df.columns

Index(['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
       'Consumer complaint narrative', 'Company public response', 'Company',
       'State', 'ZIP code', 'Tags', 'Consumer consent provided?',
       'Submitted via', 'Date sent to company', 'Company response to consumer',
       'Timely response?', 'Consumer disputed?', 'Complaint ID'],
      dtype='object')

In [4]:
df["Date received"] = pd.to_datetime(df["Date received"])

df["Date received"].min(), df["Date received"].max()

(Timestamp('2011-12-01 00:00:00'), Timestamp('2026-05-24 00:00:00'))

In [9]:
df_subset = df[
    (df["Date received"] >= "2019-01-01") &
    (df["Date received"] <= "2022-12-31")
]

df_working = df_subset.sample(
    50000,
    random_state=42
)

In [10]:
df_working.to_parquet(
    "../data/processed/complaints_50k.parquet",
    index=False
)

In [11]:
df_working.head()


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
7450420,2022-01-28,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Account information incorrect,Equifax is reporting an original creditor acco...,NaN,"EQUIFAX, INC.",GA,30327,NaN,Consent provided,Web,2022-01-28,Closed with explanation,Yes,NaN,5160432
1143839,2019-11-05,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,I am a victim of identity theft. The informati...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",CA,93722,NaN,Consent provided,Web,2019-11-05,Closed with explanation,Yes,NaN,3429041
14901371,2021-01-22,"Credit reporting, credit repair services, or o...",Credit reporting,Problem with a credit reporting company's inve...,Was not notified of investigation status or re...,NaN,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",SC,29223,NaN,Other,Web,2021-01-22,Closed with explanation,Yes,NaN,4088211
7801078,2020-09-16,"Credit reporting, credit repair services, or o...",Credit reporting,Problem with a credit reporting company's inve...,Was not notified of investigation status or re...,NaN,NaN,"EQUIFAX, INC.",AL,36605,NaN,Consent not provided,Web,2020-09-16,Closed with explanation,Yes,NaN,3848921
8716448,2022-11-30,"Credit reporting, credit repair services, or o...",Credit reporting,Improper use of your report,Reporting company used your report improperly,NaN,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,CA,92507,NaN,Consent not provided,Web,2022-11-30,Closed with explanation,Yes,NaN,6261626


In [13]:
monthly = (
    df_working.groupby(
        df_working["Date received"].dt.to_period("M")
    )
    .size()
    .reset_index(name="count")
)

monthly["Date received"] = (
    monthly["Date received"]
    .astype(str)
)

fig = px.line(
    monthly,
    x="Date received",
    y="count",
    title="Consumer Complaints Over Time"
)

fig.show()

In [14]:
top_products = (
    df_working["Product"]
    .value_counts()
    .head(10)
    .reset_index()
)

top_products.columns = ["Product", "Count"]

fig = px.bar(
    top_products,
    x="Product",
    y="Count",
    title="Top Complaint Products"
)

fig.show()

In [15]:
top_issues = (
    df_working["Issue"]
    .value_counts()
    .head(15)
    .reset_index()
)

top_issues.columns = ["Issue", "Count"]

fig = px.bar(
    top_issues,
    x="Count",
    y="Issue",
    orientation="h",
    title="Top Complaint Issues"
)

fig.show()

In [17]:
import plotly.express as px

# Top categories only
top_products = (
    df_working["Product"]
    .value_counts()
    .head(6)
    .index
)

top_issues = (
    df_working["Issue"]
    .value_counts()
    .head(12)
    .index
)

# Filter
heatmap_data = df_working[
    df_working["Product"].isin(top_products) &
    df_working["Issue"].isin(top_issues)
]

# Aggregate
heatmap_data = (
    heatmap_data.groupby(["Issue", "Product"])
    .size()
    .reset_index(name="count")
)

# Shorten long labels
heatmap_data["Issue"] = (
    heatmap_data["Issue"]
    .str.slice(0, 40)
)

heatmap_data["Product"] = (
    heatmap_data["Product"]
    .str.slice(0, 30)
)

# Plot
fig = px.density_heatmap(
    heatmap_data,
    x="Product",
    y="Issue",
    z="count",
    text_auto=True,
    title="Complaint Issues by Product"
)

fig.update_layout(
    width=1000,
    height=700,
    xaxis_title="Product",
    yaxis_title="Issue",
    font=dict(size=12)
)

fig.update_xaxes(tickangle=30)

fig.show()

In [ ]:
df_working.head()


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
7450420,2022-01-28,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Account information incorrect,Equifax is reporting an original creditor acco...,NaN,"EQUIFAX, INC.",GA,30327,NaN,Consent provided,Web,2022-01-28,Closed with explanation,Yes,NaN,5160432
1143839,2019-11-05,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,I am a victim of identity theft. The informati...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",CA,93722,NaN,Consent provided,Web,2019-11-05,Closed with explanation,Yes,NaN,3429041
14901371,2021-01-22,"Credit reporting, credit repair services, or o...",Credit reporting,Problem with a credit reporting company's inve...,Was not notified of investigation status or re...,NaN,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",SC,29223,NaN,Other,Web,2021-01-22,Closed with explanation,Yes,NaN,4088211
7801078,2020-09-16,"Credit reporting, credit repair services, or o...",Credit reporting,Problem with a credit reporting company's inve...,Was not notified of investigation status or re...,NaN,NaN,"EQUIFAX, INC.",AL,36605,NaN,Consent not provided,Web,2020-09-16,Closed with explanation,Yes,NaN,3848921
8716448,2022-11-30,"Credit reporting, credit repair services, or o...",Credit reporting,Improper use of your report,Reporting company used your report improperly,NaN,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,CA,92507,NaN,Consent not provided,Web,2022-11-30,Closed with explanation,Yes,NaN,6261626
